[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads

        self.scale = self.d_k ** -0.5

        self.kv_proj = nn.Linear(d_model, d_model*2)
        self.q_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        # self.out_drop = nn.Dropout(0.1)
        # self.attn_dropout = nn.Dropout(0.1)



        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
      B, S_q, d_model = x_q.size()
      B, S_kv, d_model = x_kv.size()
      q = self.q_proj(x_q)
      kv = self.kv_proj(x_kv)
      k, v = kv.split(d_model, dim=-1)

      q.view(B, S_q, self.num_heads, self.d_k).transpose(1, 2)
      k.view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
      v.view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)

      attn = (q @ k.transpose(-2, -1)) * self.scale

      #S_q * S_kv

      attn = attn.softmax(dim=-1)

      out = attn @ v
      out = out.transpose(1, 2).contiguous().view(B, S_q, d_model)
      out = self.out_proj(out)
      return out







In [12]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.8ms)
  ✅ [2/4] Q and KV different lengths (0.9ms)
  ✅ [3/4] No causal mask — all KV affects all Q (37.8ms)
  ✅ [4/4] Gradient flow (23.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (64.1ms total)
  Progress saved. Run status() to see your dashboard.

